# PySpark — Aggregations

Aggregations summarise many rows into fewer rows (or a single value).

| Method | Purpose |
|--------|---------|
| `groupBy()` | Group rows by one or more columns |
| `agg()` | Apply one or more aggregate functions |
| `count()` | Count rows per group |
| `sum()` | Sum a column per group |
| `avg()` / `mean()` | Average per group |
| `min()` / `max()` | Min / max per group |
| `countDistinct()` | Count unique values |
| `pivot()` | Reshape long → wide (crosstab) |

## Setup

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, min as spark_min,
    max as spark_max, countDistinct, round as spark_round
)

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("Aggregations") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    ("Alice",   "Engineering", "NY", 95000, 2023),
    ("Bob",     "Marketing",   "CA", 72000, 2023),
    ("Charlie", "Engineering", "NY", 110000, 2023),
    ("Diana",   "HR",          "TX", 65000, 2023),
    ("Eve",     "Marketing",   "CA", 85000, 2023),
    ("Frank",   "Engineering", "TX", 98000, 2022),
    ("Grace",   "HR",          "NY", 70000, 2022),
    ("Hank",    "Marketing",   "NY", 90000, 2022),
]
columns = ["name", "department", "state", "salary", "year"]
df = spark.createDataFrame(data, columns)
df.show()

+-------+-----------+-----+------+----+
|   name| department|state|salary|year|
+-------+-----------+-----+------+----+
|  Alice|Engineering|   NY| 95000|2023|
|    Bob|  Marketing|   CA| 72000|2023|
|Charlie|Engineering|   NY|110000|2023|
|  Diana|         HR|   TX| 65000|2023|
|    Eve|  Marketing|   CA| 85000|2023|
|  Frank|Engineering|   TX| 98000|2022|
|  Grace|         HR|   NY| 70000|2022|
|   Hank|  Marketing|   NY| 90000|2022|
+-------+-----------+-----+------+----+



## 1. groupBy() + count()

In [ ]:
# How many employees per department?
df.groupBy("department").count().show()

# Group by multiple columns
df.groupBy("department", "state").count().orderBy("department").show()

## 2. groupBy() + agg() — Multiple Aggregations at Once

`agg()` lets you apply several aggregate functions in a single pass — much more efficient than chaining.

In [ ]:
# Sum, average, min, max in one shot
df.groupBy("department").agg(
    count("name").alias("headcount"),
    spark_sum("salary").alias("total_salary"),
    spark_round(avg("salary"), 0).alias("avg_salary"),
    spark_min("salary").alias("min_salary"),
    spark_max("salary").alias("max_salary")
).orderBy("department").show()

## 3. countDistinct() — Count Unique Values

In [ ]:
# How many unique states does each department operate in?
df.groupBy("department").agg(
    countDistinct("state").alias("unique_states")
).show()

# Global distinct count — no groupBy
df.select(countDistinct("department").alias("num_departments")).show()

## 4. Global Aggregation — No groupBy

In [ ]:
# Total payroll, average salary, headcount across entire company
df.agg(
    count("name").alias("total_employees"),
    spark_sum("salary").alias("total_payroll"),
    spark_round(avg("salary"), 0).alias("avg_salary"),
    spark_max("salary").alias("top_salary")
).show()

## 5. Filtering After Aggregation — having() / filter()

In SQL you use `HAVING` to filter aggregated results. In PySpark you just chain `.filter()` after `.agg()`.

In [ ]:
# Departments with average salary above 85,000
df.groupBy("department") \
  .agg(spark_round(avg("salary"), 0).alias("avg_salary")) \
  .filter(col("avg_salary") > 85000) \
  .show()

# Departments with more than 2 employees
df.groupBy("department") \
  .agg(count("name").alias("headcount")) \
  .filter(col("headcount") > 2) \
  .show()

## 6. pivot() — Reshape Data (Long → Wide)

`pivot()` turns unique values of one column into separate columns — like a crosstab or Excel pivot table.

In [ ]:
# Average salary per department per year — pivoted on year
df.groupBy("department") \
  .pivot("year") \
  .agg(spark_round(avg("salary"), 0).alias("avg_salary")) \
  .show()

# Headcount per department per state
df.groupBy("department") \
  .pivot("state") \
  .agg(count("name")) \
  .fillna(0) \
  .show()

## 7. Real-World: Department Budget Report

In [ ]:
# Full department summary with % of total payroll
total_payroll = df.agg(spark_sum("salary")).collect()[0][0]

report = (
    df.groupBy("department")
    .agg(
        count("name").alias("headcount"),
        spark_sum("salary").alias("dept_payroll"),
        spark_round(avg("salary"), 0).alias("avg_salary"),
        spark_max("salary").alias("top_earner")
    )
    .withColumn("pct_of_total", spark_round(col("dept_payroll") / total_payroll * 100, 1))
    .orderBy(col("dept_payroll").desc())
)
report.show()
print(f"Company total payroll: ${total_payroll:,}")

## Quick Reference

```python
from pyspark.sql.functions import count, sum, avg, min, max, countDistinct

# Basic groupBy
df.groupBy("dept").count().show()
df.groupBy("dept", "year").count().show()

# Multiple aggregations
df.groupBy("dept").agg(
    count("id").alias("headcount"),
    sum("salary").alias("total"),
    avg("salary").alias("average"),
    min("salary").alias("lowest"),
    max("salary").alias("highest")
)

# Filter after aggregation (equivalent to SQL HAVING)
df.groupBy("dept").agg(avg("salary").alias("avg")).filter(col("avg") > 80000)

# Pivot
df.groupBy("dept").pivot("year").agg(sum("salary"))

# Global aggregation (no group)
df.agg(count("*"), sum("salary"), avg("salary"))
```

## Interview Questions

1. What is the difference between `groupBy().count()` and `df.count()`?
2. How do you apply multiple aggregate functions in one pass?
3. How do you filter aggregated results (equivalent to SQL `HAVING`)?
4. What does `pivot()` do? Give a real-world example.
5. What is `countDistinct()` and how is it different from `count()`?
6. Can you use `agg()` without `groupBy()`? What happens?
7. What is the difference between `avg()` and `mean()` in PySpark?

## Practice Exercises

**Easy:**
1. Count employees per state.
2. Find the total salary by department.
3. Find the highest-paid employee in each department.

**Medium:**
4. Find departments where average salary > 80,000 AND headcount >= 2.
5. Compute: headcount, total payroll, avg salary per department — in a single `agg()` call.
6. Pivot the data: rows = year, columns = department, values = average salary.

**Advanced:**
7. Compute each department's share (%) of total company payroll.
8. Find the department that grew most in average salary from 2022 to 2023.

In [ ]:
spark.stop()